In [2]:
import pandas as pd
import sqlite3
import os
import numpy as np

os.chdir(r"C:\Users\camde\Desktop\baseball_analytics")

conn = sqlite3.connect("data/baseball.db")

batting = pd.read_sql("SELECT * FROM batting", conn)
people = pd.read_sql("SELECT * FROM people", conn)
teams = pd.read_sql("SELECT * FROM teams", conn)
salaries = pd.read_sql("SELECT * FROM salaries", conn)
hof = pd.read_sql("SELECT * FROM hall_of_fame", conn)

print(f"Batting: {batting.shape}")
print(f"People: {people.shape}")
print(f"Salaries: {salaries.shape}")
print(f"Teams: {teams.shape}")

Batting: (128598, 22)
People: (24270, 25)
Salaries: (26428, 5)
Teams: (3614, 48)


In [2]:
batting = batting[batting['yearID'] > 1989]
salaries = salaries[salaries['yearID'] > 1989]
teams = teams[teams['yearID'] > 1989]

In [3]:
for name, df in [("batting", batting), ("people", people), ("salaries", salaries), ("teams", teams)]:
    print(f"\n── {name} ──────────────────────────")
    null_counts = df.isnull().sum().to_frame()
    print(null_counts[null_counts > 0].dropna().astype(int))


── batting ──────────────────────────
Empty DataFrame
Columns: [0]
Index: []

── people ──────────────────────────
                  0
birthYear       978
birthMonth     1352
birthDay       1521
birthCity      1130
birthCountry    939
birthState     1361
deathYear     12237
deathMonth    12244
deathDay      12341
deathCountry  12280
deathState    12352
deathCity     12300
nameFirst       413
weight         2182
height         2008
bats           2847
throws         2140
debut          3030
bbrefID        1275
finalGame      4932
retroID         897

── salaries ──────────────────────────
Empty DataFrame
Columns: [0]
Index: []

── teams ──────────────────────────
          0
DivWin   28
WCWin   134
LgWin    28
WSWin    28


In [4]:
people.drop(['deathYear', 'deathMonth', 'deathDay','deathCountry','deathState','deathCity','birthCity','birthCountry',
                     'birthState','nameFirst', 'throws', 'debut', 'finalGame', 'bbrefID','retroID', 'birthMonth', 'birthDay'], axis = 1, inplace = True)
weights_mean = people['weight'].dropna().mean()
people['weight'] = people['weight'].fillna(weights_mean)
heights_mean = people['height'].dropna().mean()
people['height'] = people['height'].fillna(heights_mean)
people['bats'] = people['bats'].fillna(people['bats'].mode()[0])
print(people['birthYear'])

0        1981.0
1        1934.0
2        1939.0
3        1954.0
4        1972.0
          ...  
24265    1939.0
24266    1958.0
24267    1924.0
24268    1888.0
24269    1990.0
Name: birthYear, Length: 24270, dtype: float64


In [5]:
for name, df in [("batting", batting), ("people", people), ("salaries", salaries), ("teams", teams)]:
    print(f"\n── {name} ──────────────────────────")
    null_counts = df.isnull().sum().to_frame()
    print(null_counts[null_counts > 0].dropna().astype(int))


── batting ──────────────────────────
Empty DataFrame
Columns: [0]
Index: []

── people ──────────────────────────
             0
birthYear  978

── salaries ──────────────────────────
Empty DataFrame
Columns: [0]
Index: []

── teams ──────────────────────────
          0
DivWin   28
WCWin   134
LgWin    28
WSWin    28


In [6]:
batting_people_df = pd.merge(batting, people, on='playerID')
df = pd.merge(batting_people_df, salaries, on = ['playerID', 'yearID'])
print(df.shape)
print(df.columns.tolist())
df['ID'].head()
df.drop(['stint', 'teamID_x', 'lgID_x', 'ID'], axis=1, inplace=True)
df.rename(columns={'teamID_y': 'teamID', 'lgID_y': 'lgID'}, inplace = True)

(24794, 32)
['playerID', 'yearID', 'stint', 'teamID_x', 'lgID_x', 'G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'IBB', 'HBP', 'SH', 'SF', 'GIDP', 'ID', 'birthYear', 'nameLast', 'nameGiven', 'weight', 'height', 'bats', 'teamID_y', 'lgID_y', 'salary']


In [7]:
obp_denom = df['AB'] + df['BB'] + df['HBP'] + df['SF']
obp_denom = obp_denom.replace(0, np.nan)
slg_denom = df['AB']
slg_denom = slg_denom.replace(0, np.nan)
df['age'] = df['yearID'] - df['birthYear']
df['OBP'] = (df['H'] + df['BB'] + df['HBP'])/(obp_denom)
df['singles'] = df['H'] - (df['2B'] + df['3B'] + df['HR'])
df['SLG'] = (df['singles'] + 2*df['2B'] + 3*df['3B'] + 4*df['HR'])/slg_denom
df['OPS'] = df['SLG'] + df['OBP']

In [8]:
df[['OBP', 'SLG', 'OPS']].describe()

,OBP,SLG,OPS
count,19178.000000,19118.000000,19118.000000
mean,0.267022,0.314498,0.579271
std,0.145139,0.199953,0.327558
min,0.000000,0.000000,0.000000
25%,0.200000,0.191489,0.398677
50%,0.303704,0.350427,0.661711
75%,0.344411,0.432432,0.772942
max,1.000000,4.000000,5.000000


In [9]:
df = df[df['AB']>=300]
df[['OBP', 'SLG', 'OPS']].describe()
df['next_OPS'] = df.groupby('playerID')['OPS'].shift(-1)
print(df['next_OPS'].isna().sum())
df = df.dropna(subset = ['next_OPS'])
print(df.shape)
print(df.columns.tolist())

1232
(4798, 34)
['playerID', 'yearID', 'G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'IBB', 'HBP', 'SH', 'SF', 'GIDP', 'birthYear', 'nameLast', 'nameGiven', 'weight', 'height', 'bats', 'teamID', 'lgID', 'salary', 'age', 'OBP', 'singles', 'SLG', 'OPS', 'next_OPS']


In [10]:
bats_dummies = pd.get_dummies(df['bats'], prefix='bats').astype(int)
df = pd.concat([df, bats_dummies], axis = 1)
df = df.drop(['teamID', 'singles', 'bats'], axis = 1)
df.to_csv('data/processed/batting_cleaned.csv', index=False)